In [ ]:
from sqlalchemy import create_engine, inspect, text
import sqlglot

In [ ]:
engine = create_engine("sqlite:///../data/db/construction.db")

In [12]:
inspector = inspect(engine)

# Получить все таблицы
tables = inspector.get_table_names()
tables

['contractors', 'objects', 'progress', 'works']

In [ ]:
# Для каждой таблицы вывести колонки
for table_name in tables:
    print(f"\n▶️ Таблица: {table_name}")
    columns = inspector.get_columns(table_name)
    for col in columns:
        print(f"  • {col['name']} | {col['type']} | nullable={col['nullable']}")

    # Внешние ключи
    fks = inspector.get_foreign_keys(table_name)
    for fk in fks:
        print(f"  ↳ FK: {fk['constrained_columns']} → {fk['referred_table']}")

    # Индексы
    indexes = inspector.get_indexes(table_name)
    for idx in indexes:
        print(f"  🔑 INDEX: {idx['name']} ({', '.join(idx['column_names'])})")


▶️ Таблица: contractors
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • work_id | INTEGER | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: objects
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • city | TEXT | nullable=False
  • budget | REAL | nullable=False

▶️ Таблица: progress
  • id | INTEGER | nullable=True
  • work_id | INTEGER | nullable=False
  • plan_vol | REAL | nullable=False
  • fact_vol | REAL | nullable=False
  • date | TEXT | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: works
  • id | INTEGER | nullable=True
  • object_id | INTEGER | nullable=False
  • work_type | TEXT | nullable=False
  • unit | TEXT | nullable=False
  ↳ FK: ['object_id'] → objects


In [14]:
with engine.connect() as conn:
    query = """
    SELECT * 
    FROM works
    JOIN contractors ON works.id = contractors.work_id
    JOIN objects ON works.object_id = objects.id
    JOIN progress ON works.id = progress.work_id
    WHERE 
        objects.name = 'ЖК Панорама 23' AND 
        objects.city = 'Санкт-Петербург' AND 
        progress.plan_vol > progress.fact_vol AND 
        contractors.name = 'ООО Новый Век'
    """

    result = conn.execute(text(query))

print(result.keys())
result.fetchall()

RMKeyView(['id', 'object_id', 'work_type', 'unit', 'id', 'name', 'work_id', 'id', 'name', 'city', 'budget', 'id', 'work_id', 'plan_vol', 'fact_vol', 'date'])


[(63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 277, 63, 271.79, 124.19, '2024-01-30'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 405, 63, 281.24, 185.88, '2024-10-16'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 368, 63, 412.6, 161.62, '2024-07-08'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 204, 63, 837.28, 793.2, '2024-11-21'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 181, 63, 903.98, 18.56, '2024-04-16')]

In [98]:
def get_schema_from_db(inspector) -> str:
    schema_parts = {}

    for table_name in inspector.get_table_names():
        columns = inspector.get_columns(table_name)
        fks = inspector.get_foreign_keys(table_name)
        schema_parts[table_name] = {"columns": columns, "foreign_keys": fks}

    return schema_parts

db_schemas = get_schema_from_db(inspector)

In [108]:
def build_system_prompt(engine) -> str:
    """Построи prompt с полным списком реальных значений из БД"""

    # Получи все уникальные значения для критичных полей
    contractors = []
    work_types = []
    cities = []

    with engine.connect() as conn:
        # Все подрядчики
        result = conn.execute(text("SELECT DISTINCT name FROM contractors"))
        contractors = [row[0] for row in result.fetchall()]

        # Все типы работ
        result = conn.execute(text("SELECT DISTINCT work_type FROM works"))
        work_types = [row[0] for row in result.fetchall()]

        # Все города
        result = conn.execute(text("SELECT DISTINCT city FROM objects"))
        cities = [row[0] for row in result.fetchall()]

    contractors_str = ", ".join([f"'{c}'" for c in contractors])
    work_types_str = ", ".join([f"'{w}'" for w in work_types])
    cities_str = ", ".join([f"'{c}'" for c in cities])

    return contractors_str, work_types_str, cities_str

In [ ]:
contractors_str, work_types_str, cities_str = build_system_prompt(engine)

LLM

In [106]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openrouter_api_key = os.getenv("OPEN_ROUTE_API_KEY")
if not openrouter_api_key:
    raise ValueError("OPEN_ROUTE_API_KEY environment variable not set.")

In [105]:
SYSTEM_PROMPT = f"""
Ты - опытный SQL разработчик, который преобразует естественные запросы в SQL.

СТРУКТУРА БД:
{db_schemas}

РЕАЛЬНЫЕ ПРИМЕРЫ ДАННЫХ:
{sample_data}

ДОСТУПНЫЕ ЗНАЧЕНИЯ В БД:
- Подрядчики: {contractors_str}
- Типы работ: {work_types_str}
- Города: {cities_str}

КРИТИЧЕСКИ ВАЖНЫЕ ПРАВИЛА:
1. ИСПОЛЬЗУЙ ТОЛЬКО ЗНАЧЕНИЯ ИЗ СПИСКА выше! Не угадывай и не изменяй названия!
2. При упоминании в запросе "подрядчик", "подрядчик" или имени - ищи точное совпадение в списке Подрядчиков
3. При упоминании "работ" или "типа работ" - ищи точное совпадение в списке Типы работ
4. При упоминании города - ищи точное совпадение в списке Города
5. ВСЕГДА используй = для точного совпадения, НИКОГДА не используй LIKE для основных фильтров
6. INNER JOIN если не сказано иное

ВЫХОД: ТОЛЬКО SQL без markdown, без объяснений!
"""

NameError: name 'contractors_str' is not defined

In [ ]:
def query_llm(query: str, system_prompt: str) -> str:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=openrouter_api_key,
    )

    completion = client.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query},
        ],
    )
    return completion.choices[0].message.content

In [101]:
user_query = """
Покажи все объекты в Петербурге, для подрядчика - Новый Век, по работам связанным с покраской. 
Резульат должен включать город, название объекта, имя подрядчика, название работы, ед.изм, 
плановый и фактический объем работ.
"""

In [102]:
sql_query_llm = query_llm(query=user_query, system_prompt=SYSTEM_PROMPT)
sql_query_llm

"SELECT o.city, o.name, c.name, w.work_type, w.unit, p.plan_vol, p.fact_vol \nFROM objects o\nJOIN works w ON o.id = w.object_id\nJOIN contractors c ON w.id = c.work_id\nJOIN progress p ON w.id = p.work_id\nWHERE o.city = 'Санкт-Петербург' AND c.name = 'ПАО МегаСтрой' AND w.work_type = 'Покраска'"

sqlglot

In [103]:
check_sql_query = sqlglot.transpile(sql_query_llm)
check_sql_query

["SELECT o.city, o.name, c.name, w.work_type, w.unit, p.plan_vol, p.fact_vol FROM objects AS o JOIN works AS w ON o.id = w.object_id JOIN contractors AS c ON w.id = c.work_id JOIN progress AS p ON w.id = p.work_id WHERE o.city = 'Санкт-Петербург' AND c.name = 'ПАО МегаСтрой' AND w.work_type = 'Покраска'"]

In [104]:
with engine.connect() as conn:
    query = check_sql_query[0]
    result = conn.execute(text(query))

print(result.keys())
result.fetchall()

RMKeyView(['city', 'name', 'name', 'work_type', 'unit', 'plan_vol', 'fact_vol'])


[]